# Основи PyTorch — навчальний проєкт

**Мета:** від тензорів і похідних перейти до повного циклу навчання нейромережі. Для прикладу використаємо **синтетичні** дані про врожайність яблук та апельсинів. Усі розрахунки запускаються на CPU.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f"PyTorch: {torch.__version__}")

## 1. Тензори

Тензор — багатовимірний масив чисел. Найважливіші властивості: `shape`, `dtype`, `device`.

In [ ]:
vector = torch.tensor([1.0, 2.0, 3.0])
matrix = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("Вектор:", vector)
print("Матриця:\n", matrix)
print("Форма матриці:", matrix.shape)
print("Матричний добуток:\n", matrix @ matrix)

## 2. Autograd

Якщо `requires_grad=True`, PyTorch будує граф операцій і може обчислити похідну. Для $f(x)=x^2+3x$ маємо $f'(x)=2x+3$. При $x=2$ похідна дорівнює 7.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3*x
y.backward()
print("f(2) =", y.item())
print("f'(2) =", x.grad.item())

## 3. Функція втрат та градієнтний спуск

Для регресії використаємо $MSE=\frac{1}{n}\sum_{i=1}^n(\hat y_i-y_i)^2$. Градієнтний спуск змінює параметри в напрямку зменшення помилки: $w_{new}=w_{old}-\eta\nabla L(w)$.

In [ ]:
weight = torch.tensor(8.0, requires_grad=True)
trajectory = [weight.item()]
learning_rate = 0.12
for epoch in range(32):
    loss = (weight - 2)**2
    loss.backward()
    with torch.no_grad():
        weight -= learning_rate * weight.grad
        weight.grad.zero_()
    trajectory.append(weight.item())

axis = np.linspace(-1, 9, 250)
plt.plot(axis, (axis - 2)**2, label="L(w) = (w − 2)²")
plt.scatter(trajectory, [(value-2)**2 for value in trajectory], s=20, label="Кроки спуску")
plt.title("Градієнтний спуск")
plt.xlabel("w")
plt.ylabel("L(w)")
plt.legend()
plt.show()

## 4. Дані та нормалізація

Ознаки `X`: температура, кількість опадів, вологість. Цілі `Y`: умовна врожайність яблук і апельсинів. Нормалізація потрібна, щоб різні масштаби ознак не погіршували навчання. **Ці дані вигадані для вправи**.

In [ ]:
features = np.array([
    [73, 67, 43], [91, 88, 64], [87, 134, 58], [102, 43, 37],
    [69, 96, 70], [78, 82, 55], [95, 110, 60], [85, 75, 48],
    [100, 120, 65], [71, 56, 42], [89, 90, 51], [98, 101, 72]
], dtype=np.float32)
targets = np.array([
    [56, 70], [81, 101], [119, 89], [22, 39],
    [104, 95], [78, 82], [98, 91], [69, 75],
    [110, 94], [46, 62], [77, 85], [104, 107]
], dtype=np.float32)

X = torch.from_numpy(features)
Y = torch.from_numpy(targets)
x_mean, x_std = X.mean(dim=0), X.std(dim=0, unbiased=False)
y_mean, y_std = Y.mean(dim=0), Y.std(dim=0, unbiased=False)
X_norm = (X - x_mean) / x_std
Y_norm = (Y - y_mean) / y_std
print("X:", X_norm.shape, "Y:", Y_norm.shape)

## 5. Лінійна модель через `nn.Linear`

Модель обчислює $\hat{y} = XW+b$. Три входи, два виходи; параметри підбирає оптимізатор `SGD`.

In [ ]:
torch.manual_seed(SEED)
linear_model = nn.Linear(3, 2)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(linear_model.parameters(), lr=0.05)
linear_losses = []
for epoch in range(600):
    predictions = linear_model(X_norm)
    loss = criterion(predictions, Y_norm)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    linear_losses.append(loss.item())

with torch.no_grad():
    linear_predictions = linear_model(X_norm) * y_std + y_mean
print(f"Остаточна MSE (нормалізована): {linear_losses[-1]:.4f}")

## 6. Нейромережа 3 → 3 → 2 з ReLU

Нелінійність `ReLU(x)=max(0,x)` дозволяє мережі моделювати залежності складніші за лінійні. Архітектура: 3 ознаки → 3 приховані нейрони → 2 прогнози.

In [ ]:
torch.manual_seed(SEED)
neural_model = nn.Sequential(
    nn.Linear(3, 3),
    nn.ReLU(),
    nn.Linear(3, 2)
)
optimizer_nn = torch.optim.Adam(neural_model.parameters(), lr=0.03)
neural_losses = []
for epoch in range(1000):
    predictions = neural_model(X_norm)
    loss = criterion(predictions, Y_norm)
    optimizer_nn.zero_grad()
    loss.backward()
    optimizer_nn.step()
    neural_losses.append(loss.item())

with torch.no_grad():
    neural_predictions = neural_model(X_norm) * y_std + y_mean
print(f"Остаточна MSE (нормалізована): {neural_losses[-1]:.4f}")

## 7. Візуалізація

Порівнюємо втрати, фактичні та передбачені значення, а також MSE двох моделей **на тих самих навчальних даних**. Це демонстрація процесу, а не незалежна оцінка узагальнювальної здатності.

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(linear_losses, label="Лінійна модель")
plt.plot(neural_losses, label="Мережа 3-3-2")
plt.xlabel("Епоха")
plt.ylabel("MSE")
plt.title("Динаміка функції втрат")
plt.legend()
plt.show()

for index, crop in enumerate(["яблук", "апельсинів"]):
    plt.figure(figsize=(8, 4.5))
    plt.plot(Y[:, index].numpy(), "o-", label="Факт")
    plt.plot(neural_predictions[:, index].numpy(), "s--", label="Прогноз")
    plt.title(f"Прогноз врожайності {crop}")
    plt.xlabel("Індекс спостереження")
    plt.ylabel("Умовна врожайність")
    plt.legend()
    plt.show()

In [ ]:
linear_mse = torch.mean((linear_predictions - Y)**2).item()
neural_mse = torch.mean((neural_predictions - Y)**2).item()

plt.bar(["Лінійна", "Нейромережа"], [linear_mse, neural_mse])
plt.ylabel("MSE у вихідній шкалі")
plt.title("Порівняння на навчальній вибірці")
plt.show()
print(f"MSE лінійної моделі: {linear_mse:.3f}")
print(f"MSE нейромережі: {neural_mse:.3f}")

## 8. Висновки та наступні кроки

- `autograd` сам знаходить похідні для обчислення градієнтів.
- `nn.Linear` реалізує лінійну регресію з навчальними параметрами.
- `ReLU` надає нейромережі нелінійності.
- Втрати на тренувальних даних **не дорівнюють** якості на нових спостереженнях.
- Для реального проєкту потрібні більший набір даних, розділення train/validation/test, метрики на відкладеній вибірці та контроль перенавчання.

**Завдання для практики:** змініть кількість прихованих нейронів, швидкість навчання та оптимізатор; порівняйте криві втрат.